# E07 — Evidence-aware multi-objective material selection
This notebook independently reproduces the executed JavaScript analysis using committed source CSVs. It is supplied unexecuted; the saved numerical results were generated by the companion JavaScript analysis. Thresholds are illustrative, not safety specifications. Missing local data remain missing.


In [ ]:
from pathlib import Path
import subprocess, pandas as pd, numpy as np
ROOT=Path.cwd()
if not (ROOT/'outputs/E07/audit.json').exists():
    if (ROOT.parent/'outputs/E07/audit.json').exists(): ROOT=ROOT.parent
    else:
        ROOT=Path('/content/SeaweedMat')
        if not ROOT.exists(): subprocess.run(['git','clone','https://github.com/peterDataScientia/SeaweedMat.git',str(ROOT)],check=True)
read=lambda p: pd.read_csv(ROOT/p)
m=read('outputs/E04/primary_mechanical_comparison.csv')
b=read('outputs/E05/theobald_summary.csv')
e=read('outputs/E06/replacement_scenarios.csv')
assert (len(m),len(b),len(e))==(9,10,60)
def front(df,cols):
    a=df[cols].to_numpy(float)
    return np.array([not any(np.all(y>=x) and np.any(y>x) for y in a) for x in a])
m['model_low_retention_pct']=m[['interpolated_retention_pct','model_min_pct']].min(axis=1)
m['model_low_force_N']=m.initial_value*9.80665*m.model_low_retention_pct/100
nom=['interpolated_retention_pct','breaking_force_N']
low=['model_low_retention_pct','model_low_force_N']
m['front']=front(m,nom)
m['low_front']=front(m,low)
c=m[m.material=='Cotton'].copy()
c['front']=front(c,nom)
c['low_front']=front(c,low)
assert m.loc[m.front,'label'].tolist()==['Twisted jute 6']
assert m.loc[m.low_front,'label'].tolist()==['Twisted jute 6']
assert set(c.loc[c.front,'label'])==set(c.loc[c.low_front,'label'])=={'Twisted cotton 3','Braided cotton 5'}
display(m[['label']+nom+low+['front','low_front']])


In [ ]:
s=[]
for r in [60,70,80,90,100]:
    for f in [100,200,300,400,600]:
        n=m[(m[nom[0]]>=r)&(m[nom[1]]>=f)]
        l=m[(m[low[0]]>=r)&(m[low[1]]>=f)]
        s.append(dict(min_retention_pct=r,min_breaking_force_N=f,nominal_count=len(n),model_low_count=len(l),nominal_candidates=';'.join(n.label),model_low_candidates=';'.join(l.label)))
S=pd.DataFrame(s)
ref=read('outputs/E07/constraint_scenarios.csv')
keys=['min_retention_pct','min_breaking_force_N']
j=S.merge(ref,on=keys,validate='one_to_one',suffixes=('_rerun','_saved'))
for k in ['nominal_count','model_low_count']:
    np.testing.assert_array_equal(j[k+'_rerun'],j[k+'_saved'])
assert (S.nominal_count>0).sum()==25 and (S.model_low_count>0).sum()==20
display(S)
B=[]
for coat in [False,True]:
    sub=b[b.treatment.str.endswith(' + alginate').eq(coat)&~b.treatment.str.startswith('Control')].copy()
    sub['material']=sub.treatment.str.replace(' + alginate','',regex=False)
    sub['coating']='alginate' if coat else 'uncoated'
    sub['pareto']=front(sub,['density_cm2_mean','combined_mm_mean','longest_mm_mean'])
    sub['pareto_two']=front(sub,['density_cm2_mean','combined_mm_mean'])
    assert set(sub.loc[sub.pareto,'material'])==set(sub.loc[sub.pareto_two,'material'])=={'Cotton','Polypropylene'}
    B.append(sub)
display(pd.concat(B)[['material','coating','density_cm2_mean','combined_mm_mean','longest_mm_mean','pareto','pareto_two']])


In [ ]:
e['calculated_cost_ratio']=e.price_per_m_ratio/(e.lifetime_ratio*e.marketable_yield_ratio)
e['maximum_price_or_impact_ratio']=e.lifetime_ratio*e.marketable_yield_ratio
e['cost_not_higher']=e.calculated_cost_ratio<=1+1e-10
np.testing.assert_allclose(e.calculated_cost_ratio,e.material_cost_per_output_ratio,atol=1e-7,rtol=0)
assert e.cost_not_higher.sum()==26
evidence=read('outputs/E07/integrated_decision_matrix.csv')
assert len(evidence)==8 and evidence.full_selection_status.eq('not evaluable').all()
display(evidence[['material','trial_priority','full_selection_status']])
print('E07 checks passed. Scenario counts are not probabilities.')


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.weight':'bold','axes.labelweight':'bold','axes.labelsize':15,'font.size':12,'axes.linewidth':2,'xtick.major.width':2,'ytick.major.width':2,'xtick.major.size':6,'ytick.major.size':6,'axes.spines.top':False,'axes.spines.right':False})
out=ROOT/'outputs/E07_python_rerun'
out.mkdir(parents=True,exist_ok=True)
fig,ax=plt.subplots(figsize=(150/25.4,110/25.4))
for mat,color in [('Cotton','#008D79'),('Jute','#075DE8')]:
    p=m[m.material==mat]
    ax.scatter(p.interpolated_retention_pct,p.breaking_force_N,label=mat,color=color,s=55)
for _,r in m[m.front].iterrows():
    ax.annotate(r.label,(r.interpolated_retention_pct,r.breaking_force_N),xytext=(-8,-18),textcoords='offset points',ha='right',fontsize=10)
ax.set_xlabel('Strength retention [%]')
ax.set_ylabel('Breaking force [N]')
ax.legend(frameon=False)
fig.tight_layout()
for ext in ['png','svg','pdf']: fig.savefig(out/('mechanical_pareto.'+ext),dpi=600,bbox_inches='tight',facecolor='white')
plt.show()
